In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from mlxtend.evaluate import bias_variance_decomp

In [2]:
RANDOM_STATE = 42

In [3]:
data = fetch_california_housing(as_frame = True)

X = data.data
y = data.target

In [4]:
X.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [5]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE)

In [6]:
scaler = StandardScaler()
scaler.fit(Xtrain)

Xtrain = pd.DataFrame(data = scaler.transform(Xtrain), columns = X.columns)
Xtest = pd.DataFrame(data = scaler.transform(Xtest), columns = X.columns)

# Bias Variance Decomposition

В библиотеке `mlxtend` есть функция `bias_variance_decomp` для оценки компонент разложения ошибки.

### Гиперпараметры
* `estimator` - семейство моделей
* `X_train, y_train` - обучающие данные
* `X_test, y_test` - тестовые данные
* `loss` - функция потерь (MSE для регрессии, 0-1 loss (доля ошибок модели) для классификации)
* `num_rounds=200` - число случайных подвыборок из `X_train` для обучения модели

### Возвращаемые значения

* `avg_expected_loss` - ошибка на тестовых данных
* `avg_expected_bias` - смещение
* `avg_expected_variance` - разброс

Функция работает с объектами `np.array` => переведем наши объекты к этому типу

In [7]:
X_train = Xtrain.values
X_test = Xtest.values
y_train = ytrain.values
y_test = ytest.values

In [8]:
avg_mse, avg_bias, avg_var = bias_variance_decomp(LinearRegression(), X_train, y_train, X_test, y_test, loss = 'mse', 
                                                  random_seed=np.random.seed(RANDOM_STATE))

In [9]:
print('Loss:', avg_mse)
print('Bias:', avg_bias)
print('Variance:', avg_var)

Loss: 0.5315377404530977
Bias: 0.5290843972900264
Variance: 0.002453343163071313


In [10]:
avg_mse, avg_bias, avg_var = bias_variance_decomp(DecisionTreeRegressor(), X_train, y_train, X_test, y_test, loss = 'mse', 
                                                  random_seed=np.random.seed(RANDOM_STATE))

In [11]:
print('Loss:', avg_mse)
print('Bias:', avg_bias)
print('Variance:', avg_var)

Loss: 0.5712774893090806
Bias: 0.2552897159337192
Variance: 0.31598777337536144


Мы видим, что решающее дерево гораздо точнее предсказывает целевую переменную, чем линейная регрессия  
(*bias* гораздо меньше), но при этом гораздо сильнее переобучено (*variance* гораздо больше).  
За счет этого суммарная ошибка у дерева чуть больше, чем у линейной регрессии.

Путем подбора гиперпараметров дерева можно снизить переобучение и суммарную ошибку.